# 05 — Model Evaluation (Pre-Hypothesis Testing)

**Primary author:** Victoria

**Builds on:**
- *03_train_g1.ipynb* (Victoria — produced the training and validation triplet files used for the generalization check in §2)
- *04_embedding_verification.ipynb* (Victoria — confirmed all four validation embedding sets are shape-correct, non-degenerate, and index-aligned; this notebook relies on those guarantees)
- *notebooks/archive/09_learned_g_misdirection.ipynb* (Victoria — the earlier `g1_tokenspan` analysis whose collapse symptom motivated §3)
- *DECISIONS.md §20* (mean pooling is canonical; `g1` in this notebook refers to the mean-pooled model, not the tokenspan variant)

**Prompt engineering:** Victoria
**AI assistance:** Claude / Claude Code (Anthropic)
**Environment:** Local

---

Before we compute and interpret the ATE in the Stage 6 hypothesis test, we
need to be sure we understand the basic health of `g1`. Three things can
silently invalidate an ATE comparison:

1. **The model did not generalize the training objective.** Low-quality
   fine-tuning can memorize training triplets without transferring to unseen
   ones; the resulting embeddings may look "trained" without being useful.
2. **The embedding space collapsed.** The earlier `g1_tokenspan` run (NB 09)
   compressed `f_common_wndef` embeddings indiscriminately — the space got
   tighter globally rather than more discriminating locally. An ATE computed
   over a collapsed space is not evidence of misdirection reduction.
3. **The two ATE components moved in unexpected ways.** The ATE is a
   difference of two cosines; only looking at the difference hides whether
   T=0 and T=1 are both changing, or only one.

This notebook answers those three questions, plus an RSA summary of how
much the overall similarity structure changed. Everything runs locally on
CPU from pre-computed validation embeddings — no new GPU work.

**Scope.** The canonical mean-pooling models only (`g_stock` and `g1`).
The `_tokenspan` variants are out of scope per Decision 20; they are kept
around only for the verification check in NB 04.

---

## §0 — Imports and configuration

Environment auto-detection for Local / Great Lakes / Colab. A version-
reporting cell follows (Decision 18) — this prints rather than asserts so
that a collaborator with a slightly different environment can still run
the notebook, but any mismatch is visible at a glance.

In [ ]:
# ============================================================
# Imports and configuration
# ============================================================
import time
from datetime import date
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# --- Environment auto-detection ---
try:
    IS_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IS_COLAB = False

IS_GREATLAKES = Path("/nfs/turbo").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Research Project - NLP CCC's/ccc-project")
elif IS_GREATLAKES:
    PROJECT_ROOT = Path.home() / "ccc-project"
else:
    PROJECT_ROOT = Path("../..").resolve()

COMPONENT_ROOT  = PROJECT_ROOT / "custom_embedding_model"
EMBEDDINGS_DIR  = COMPONENT_ROOT / "data" / "embeddings"
WN_DIR          = COMPONENT_ROOT / "data" / "filtered_split" / "wn_synset"
TRIPLETS_DIR    = COMPONENT_ROOT / "data" / "triplets"
OUTPUT_DIR      = COMPONENT_ROOT / "outputs"
FIG_DIR         = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

env_label = "Colab" if IS_COLAB else ("Great Lakes" if IS_GREATLAKES else "Local")
print(f"Environment:     {env_label}")
print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"EMBEDDINGS_DIR:  {EMBEDDINGS_DIR}")
print(f"TRIPLETS_DIR:    {TRIPLETS_DIR}")
print(f"OUTPUT_DIR:      {OUTPUT_DIR}")

# Pin the RNG once. §3 and §5 sample pairs / vocabulary subsets; fixing the
# seed keeps those samples identical across re-runs.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Version reporting (Decision 18). Printed, not asserted.
import matplotlib, seaborn
print()
print(f"pandas:     {pd.__version__}")
print(f"numpy:      {np.__version__}")
print(f"scipy:      {scipy.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn:    {seaborn.__version__}")

---

## §1 — Load embeddings and index files

We load validation embeddings for just the two canonical models, `g_stock`
and `g1` — three phrase types each (`f_clue_val`, `f_common_wndef_val`,
`f_common_wnex_val`), six `.npy` arrays in total. Alongside the arrays we
load:

- The `f_clue_val_index.csv` that maps `(clue_id, definition)` to row
  positions in `f_clue_val.npy`. NB 04 confirmed the two models' indexes
  are identical, so we use `g_stock`'s copy as the single source of truth.
- The two validation-split vocabulary files that index the
  `f_common_*_val.npy` arrays.
- `clues_val.csv` — the canonical validation rows we need for §4's
  (clue, definition, answer) evaluation pairs.

Expected shapes from NB 04 / FINDINGS.md:
`f_clue_val (47933, 1024)`, `f_common_wndef_val (26152, 1024)`,
`f_common_wnex_val (3008, 1024)`. We assert these after loading.

All CSVs load with `keep_default_na=False, na_values=[""]` because `"nan"`
(grandmother) is a valid crossword word and must not be coerced.

In [ ]:
# ============================================================
# Model / phrase-type registry
# ============================================================
MODEL_NAMES  = ["g_stock", "g1"]
PHRASE_TYPES = ["f_clue_val", "f_common_wndef_val", "f_common_wnex_val"]

EXPECTED_SHAPE = {
    "f_clue_val":         (47933, 1024),
    "f_common_wndef_val": (26152, 1024),
    "f_common_wnex_val":  (3008, 1024),
}

In [ ]:
# ============================================================
# Load the six .npy arrays + index / vocabulary / clue CSVs
# ============================================================
t0 = time.time()

# embeddings[(model, phrase)] -> np.ndarray of shape (N, 1024)
embeddings = {}
for model in MODEL_NAMES:
    for phrase in PHRASE_TYPES:
        emb = np.load(EMBEDDINGS_DIR / model / f"{phrase}.npy")
        # Shape validation per CLAUDE.md — catches any accidental swap of a
        # wndef file for a wnex file (different row counts).
        assert emb.shape == EXPECTED_SHAPE[phrase], (
            f"{model}/{phrase}: shape {emb.shape} != expected {EXPECTED_SHAPE[phrase]}"
        )
        embeddings[(model, phrase)] = emb

# Shared f_clue index (identical across models per NB 04's consistency check).
f_clue_index = pd.read_csv(
    EMBEDDINGS_DIR / "g_stock" / "f_clue_val_index.csv",
    keep_default_na=False, na_values=[""],
)

# Vocabulary files — row column is the canonical index into f_common_*_val.npy.
vocab_wndef_val = pd.read_csv(
    WN_DIR / "wndef" / "vocabulary_wndef_val.csv",
    keep_default_na=False, na_values=[""],
)
vocab_wnex_val = pd.read_csv(
    WN_DIR / "wnex" / "vocabulary_wnex_val.csv",
    keep_default_na=False, na_values=[""],
)

# Validation clue rows, for the (clue, definition, answer) evaluation pairs in §4.
clues_val = pd.read_csv(
    WN_DIR / "clues_val.csv",
    keep_default_na=False, na_values=[""],
)

load_seconds = time.time() - t0
print(f"Loaded 6 .npy + 4 CSVs in {load_seconds:.1f}s")
print(f"  vocabulary_wndef_val: {len(vocab_wndef_val):,} rows")
print(f"  vocabulary_wnex_val:  {len(vocab_wnex_val):,} rows")
print(f"  f_clue_val_index:     {len(f_clue_index):,} rows")
print(f"  clues_val:            {len(clues_val):,} rows")

# Cross-check .npy row counts against the corresponding vocabulary / index
# file lengths, separate from the EXPECTED_SHAPE constant check above.
for model in MODEL_NAMES:
    assert embeddings[(model, "f_common_wndef_val")].shape[0] == len(vocab_wndef_val)
    assert embeddings[(model, "f_common_wnex_val")].shape[0] == len(vocab_wnex_val)
    assert embeddings[(model, "f_clue_val")].shape[0] == len(f_clue_index)
print("All row-count cross-checks pass.")

In [ ]:
# ============================================================
# Lookup dicts: word / clue-key -> row index
# ============================================================
# Building these once here avoids re-scanning CSVs in every later section.
wndef_word_to_row = dict(zip(vocab_wndef_val["word"], vocab_wndef_val["row"]))
wnex_word_to_row  = dict(zip(vocab_wnex_val["word"],  vocab_wnex_val["row"]))

# f_clue is indexed by the composite (clue_id, definition) key — use it as a tuple.
clue_key_to_row = {
    (cid, defn): row
    for cid, defn, row in zip(
        f_clue_index["clue_id"], f_clue_index["definition"], f_clue_index["row"]
    )
}

print(f"wndef_word_to_row: {len(wndef_word_to_row):,} entries")
print(f"wnex_word_to_row:  {len(wnex_word_to_row):,} entries")
print(f"clue_key_to_row:   {len(clue_key_to_row):,} entries")

In [ ]:
# ============================================================
# Rowwise cosine helper (matches the DATA.md definition)
# ============================================================
def rowwise_cosine(A, B):
    """Return per-row cosine similarity between two equal-shape (N, D) arrays.

    Identical to the helper used in NB 04 and documented in DATA.md.
    """
    # +1e-10 guard against accidental zero rows. NB 04 already checked for
    # zero rows in all six of our arrays, so this should never fire.
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-10)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-10)
    return np.sum(A_norm * B_norm, axis=1)

---

## §2 — Validation triplet accuracy

This is the most direct overfitting diagnostic we have: the triplet
constraint the model was trained on, evaluated on triplets the model has
never seen. `data/triplets/g1_val.csv` was constructed in NB 03 using the
identical pipeline to `g1_train.csv` — same distractor source, same phrase
lookups — so differences in triplet accuracy between train and validate
reflect generalization, not construction artifacts.

For each validation triplet, we resolve three row indices (anchor,
positive, negative), pull the corresponding pre-computed embeddings, and
check whether `cos(anchor, positive) > cos(anchor, negative)` — the
satisfied-triplet condition. We report this both for `g1` and for `g_stock`;
`g_stock` is the untrained baseline, so its accuracy tells us how well the
stock model already separates positives from negatives. A large gap
`g1 − g_stock` is evidence that the fine-tuning learned a generalizable
signal; a small or negative gap is evidence of poor generalization.

**Bias caveat** (see DECISIONS.md Decision 21): ~41% of validation
triplets are dropped because their `distractor_wn` values are absent from
`vocabulary_wndef_val.csv` — distractors are drawn from the full WordNet
vocabulary, but validation-split embeddings cover only words appearing as
definitions or answers in validation-split clues (Decision 8). The
surviving triplets' negatives are words that also appear as definitions or
answers in validation clues — i.e., common crossword words. Whether these
are systematically easier or harder negatives than the dropped distractors
is unknown. However, the comparison between `g_stock` and `g1` is computed
on the identical set of triplets, so any difficulty bias affects both
models equally and does not compromise the `g_stock`-vs-`g1` comparison.

In [ ]:
# ============================================================
# §2a — Load validation triplets and resolve embedding rows
# ============================================================
val_triplets = pd.read_csv(
    TRIPLETS_DIR / "g1_val.csv",
    keep_default_na=False, na_values=[""],
)
print(f"Loaded {len(val_triplets):,} validation triplets from g1_val.csv")
print(f"\nColumns: {list(val_triplets.columns)}")

# Resolve row indices for each of the three triplet roles. A miss returns -1
# so we can vectorize the drop step.
anchor_rows = np.array([
    clue_key_to_row.get((cid, defn), -1)
    for cid, defn in zip(val_triplets["clue_id"], val_triplets["definition"])
])
positive_rows = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["answer_wn"]
])
negative_rows = np.array([
    wndef_word_to_row.get(w, -1) for w in val_triplets["distractor_wn"]
])

n_total = len(val_triplets)
mask = (anchor_rows >= 0) & (positive_rows >= 0) & (negative_rows >= 0)
n_valid = int(mask.sum())

# Per-role resolution counts so we know which role is losing rows.
n_miss_anchor   = int((anchor_rows   < 0).sum())
n_miss_positive = int((positive_rows < 0).sum())
n_miss_negative = int((negative_rows < 0).sum())

print(f"\nResolution counts:")
print(f"  anchors resolved:   {n_total - n_miss_anchor:,} / {n_total:,}")
print(f"  positives resolved: {n_total - n_miss_positive:,} / {n_total:,}")
print(f"  negatives resolved: {n_total - n_miss_negative:,} / {n_total:,}")
print(f"  all three resolved: {n_valid:,} / {n_total:,} "
      f"({n_valid / n_total:.1%})")

# Diagnostic note: distractors come from dataset_harder.parquet and are drawn
# from the full WordNet vocabulary, while vocabulary_wndef_val.csv only covers
# words that appear as def/answer in the VALIDATION split. Per Decision 8, g1
# embeddings cover the validation split only during iteration — so distractors
# that are only seen in the training or test split have no g1 (or g_stock_val)
# row to resolve to. The surviving subset is still large enough for a
# meaningful triplet-accuracy reading, but the coverage loss is documented
# here and in planning/questions/05_model_evaluation-questions.md.
if n_miss_negative > 0:
    print(f"\nNOTE: {n_miss_negative:,} validation triplets have a "
          f"distractor_wn not present in vocabulary_wndef_val.csv "
          f"(the validation-split wndef vocabulary). This is expected — "
          f"distractors are drawn from the full WordNet vocabulary, not "
          f"restricted to validation-split words. These triplets are "
          f"dropped from the §2 evaluation. See "
          f"planning/questions/05_model_evaluation-questions.md for "
          f"discussion and options.")

# Shape assertion: anchors and positives should resolve 100% because they come
# from validation-split clues / answers, both of which are fully covered by
# f_clue_val and vocabulary_wndef_val.
assert n_miss_anchor == 0,   f"{n_miss_anchor:,} anchors failed to resolve"
assert n_miss_positive == 0, f"{n_miss_positive:,} positives failed to resolve"

# Keep only fully-resolved rows for the downstream accuracy computation.
anchor_rows   = anchor_rows[mask]
positive_rows = positive_rows[mask]
negative_rows = negative_rows[mask]

In [ ]:
# ============================================================
# §2b — Triplet accuracy for g_stock and g1
# ============================================================
def triplet_stats(model):
    """Return a dict of triplet accuracy / margin statistics for one model."""
    clue_emb  = embeddings[(model, "f_clue_val")]
    wndef_emb = embeddings[(model, "f_common_wndef_val")]
    # Gather per-triplet embeddings in one indexing pass each.
    A = clue_emb[anchor_rows]
    P = wndef_emb[positive_rows]
    N = wndef_emb[negative_rows]
    cos_pos = rowwise_cosine(A, P)
    cos_neg = rowwise_cosine(A, N)
    margin  = cos_pos - cos_neg
    return {
        "model":        model,
        "n_triplets":   len(margin),
        "accuracy":     float((margin > 0).mean()),
        "mean_margin":  float(margin.mean()),
        "median_margin": float(np.median(margin)),
        "pct_margin_over_0.1": float((margin > 0.1).mean()),
        "pct_margin_over_0.5": float((margin > 0.5).mean()),
        # Store the raw margin vector for plotting in §2c.
        "_margin": margin,
        "_cos_pos": cos_pos,
        "_cos_neg": cos_neg,
    }

stats_stock = triplet_stats("g_stock")
stats_g1    = triplet_stats("g1")

# Render a side-by-side table. We keep private keys (prefixed _) out of it.
triplet_table = pd.DataFrame([
    {"Metric": "Triplet accuracy (% correct)",
     "g_stock": stats_stock["accuracy"] * 100,
     "g1":      stats_g1["accuracy"]    * 100},
    {"Metric": "Mean margin (cos_pos - cos_neg)",
     "g_stock": stats_stock["mean_margin"],
     "g1":      stats_g1["mean_margin"]},
    {"Metric": "Median margin",
     "g_stock": stats_stock["median_margin"],
     "g1":      stats_g1["median_margin"]},
    {"Metric": "% triplets with margin > 0.1",
     "g_stock": stats_stock["pct_margin_over_0.1"] * 100,
     "g1":      stats_g1["pct_margin_over_0.1"]    * 100},
    {"Metric": "% triplets with margin > 0.5",
     "g_stock": stats_stock["pct_margin_over_0.5"] * 100,
     "g1":      stats_g1["pct_margin_over_0.5"]    * 100},
    {"Metric": "N validation triplets evaluated",
     "g_stock": stats_stock["n_triplets"],
     "g1":      stats_g1["n_triplets"]},
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(triplet_table.to_string(index=False))

In [ ]:
# ============================================================
# §2c — Margin distribution histogram
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))

# Use a common bin edge set so the two histograms are directly comparable.
both = np.concatenate([stats_stock["_margin"], stats_g1["_margin"]])
bins = np.linspace(both.min(), both.max(), 80)

ax.hist(stats_stock["_margin"], bins=bins, alpha=0.55, label="g_stock",
        color="tab:blue")
ax.hist(stats_g1["_margin"],    bins=bins, alpha=0.55, label="g1",
        color="tab:orange")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--",
           label="margin = 0 (correct / incorrect boundary)")
ax.axvline(stats_stock["_margin"].mean(), color="tab:blue",  linewidth=1.2,
           linestyle=":")
ax.axvline(stats_g1["_margin"].mean(),    color="tab:orange", linewidth=1.2,
           linestyle=":")
ax.set_xlabel("cos(anchor, positive) - cos(anchor, negative)")
ax.set_ylabel("Count of validation triplets")
ax.set_title("Validation triplet margin distribution (g_stock vs g1)")
ax.legend()
fig.tight_layout()

fig_path = FIG_DIR / "05_val_triplet_accuracy.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §3 — Collapse detection

A healthy embedding space discriminates: semantically distinct words sit
far apart, semantically related ones close together. A collapsed space
pulls everything together indiscriminately — the geometry compresses and
cosine-based downstream tasks lose their signal.

We test for collapse two ways:

1. **Mean pairwise cosine among random word pairs.** If `g1` compresses
   the space, arbitrary pairs should look more similar under `g1` than
   under `g_stock`.
2. **Effective dimensionality (participation ratio).** A collapsed space
   concentrates variance into a few principal directions; a healthy space
   spreads it across many. The participation ratio `(Σs²)² / Σs⁴` maps
   that intuition onto a single number bounded between 1 (maximally
   collapsed) and D (perfectly isotropic).

The `g1_tokenspan` run in NB 09 showed both symptoms on `f_common_wndef`.
Whether the mean-pooled `g1` shows them is an open question — the training
objective is similar, but the pooling method differs.

In [ ]:
# ============================================================
# §3a — Mean pairwise cosine among random word pairs
# ============================================================
# Use the same sampled pair indices for both models within a phrase type so
# the two numbers are directly comparable.
N_PAIRS = 50_000

def sample_pairs(n_rows, n_pairs, seed):
    """Return two int arrays (i, j) of length n_pairs, with i != j on every row.

    Sampling with replacement from the product is fine for 50k pairs out of
    at least C(3008, 2) = ~4.5M candidates — the collision rate is negligible,
    and any stray (i, i) pair gets resampled below.
    """
    rng = np.random.default_rng(seed)
    i = rng.integers(0, n_rows, size=n_pairs)
    j = rng.integers(0, n_rows, size=n_pairs)
    # Resample any self-pair until clean. In practice this resolves in one pass.
    same = i == j
    while same.any():
        j[same] = rng.integers(0, n_rows, size=int(same.sum()))
        same = i == j
    return i, j

pairwise_rows = []
pair_sims = {}  # keyed by (model, phrase) for §3c's histogram
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    # Same (i, j) pairs across the two models for this phrase type.
    n_rows_phrase = embeddings[(MODEL_NAMES[0], phrase)].shape[0]
    i_idx, j_idx = sample_pairs(n_rows_phrase, N_PAIRS, seed=RANDOM_STATE)

    for model in MODEL_NAMES:
        emb = embeddings[(model, phrase)]
        A = emb[i_idx]
        B = emb[j_idx]
        sims = rowwise_cosine(A, B)
        pair_sims[(model, phrase)] = sims
        pairwise_rows.append({
            "Model":        model,
            "Phrase type":  phrase,
            "Mean":         float(sims.mean()),
            "Median":       float(np.median(sims)),
            "Std":          float(sims.std()),
            "P5":           float(np.percentile(sims, 5)),
            "P95":          float(np.percentile(sims, 95)),
        })

pairwise_df = pd.DataFrame(pairwise_rows)
with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(pairwise_df.to_string(index=False))

In [ ]:
# ============================================================
# §3b — Embedding variance and effective dimensionality
# ============================================================
def variance_stats(emb):
    """Total variance, participation ratio, and cumulative variance fractions.

    We center the embedding matrix and take its singular values. Squared
    singular values are proportional to variance per principal component.
    Participation ratio = (sum(s^2))^2 / sum(s^4) — 1 when all variance sits
    in a single component, D when it is perfectly spread across D dims.
    """
    centered = emb - emb.mean(axis=0, keepdims=True)
    # full_matrices=False keeps SVD in the economy form — still exact, much cheaper.
    s = np.linalg.svd(centered, full_matrices=False, compute_uv=False)
    s2 = s ** 2
    total_var = float(s2.sum())
    # Participation ratio (effective dimensionality).
    eff_dim = float(total_var ** 2 / np.sum(s ** 4))
    cum = np.cumsum(s2) / total_var
    return {
        "total_var": total_var,
        "eff_dim":   eff_dim,
        "top10_frac":  float(cum[9]),
        "top50_frac":  float(cum[49]),
        "top100_frac": float(cum[99]),
        "_cum": cum,
    }

variance_rows = []
variance_detail = {}  # (model, phrase) -> dict incl. _cum for §3c
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    for model in MODEL_NAMES:
        stats = variance_stats(embeddings[(model, phrase)])
        variance_detail[(model, phrase)] = stats
        variance_rows.append({
            "Model":         model,
            "Phrase type":   phrase,
            "Total var":     stats["total_var"],
            "Eff. dim":      stats["eff_dim"],
            "Top-10 var %":  stats["top10_frac"]  * 100,
            "Top-50 var %":  stats["top50_frac"]  * 100,
            "Top-100 var %": stats["top100_frac"] * 100,
        })

variance_df = pd.DataFrame(variance_rows)
with pd.option_context("display.float_format", "{:.2f}".format,
                       "display.width", 140):
    print(variance_df.to_string(index=False))

In [ ]:
# ============================================================
# §3c — Collapse visualizations
# ============================================================
# Figure 1: overlaid pairwise-cosine histograms, one panel per phrase type.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, phrase in zip(axes, ["f_common_wndef_val", "f_common_wnex_val"]):
    # Common bins so g_stock and g1 histograms are directly comparable.
    both = np.concatenate([pair_sims[("g_stock", phrase)],
                           pair_sims[("g1",      phrase)]])
    bins = np.linspace(both.min(), both.max(), 70)
    for model, color in [("g_stock", "tab:blue"), ("g1", "tab:orange")]:
        sims = pair_sims[(model, phrase)]
        ax.hist(sims, bins=bins, alpha=0.55, color=color,
                label=f"{model} (mean={sims.mean():.3f})")
    ax.set_title(f"Random pairwise cosine — {phrase}")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("count")
    ax.legend()
fig.suptitle(f"Pairwise cosine among {N_PAIRS:,} random word pairs")
fig.tight_layout()
fig_path = FIG_DIR / "05_collapse_pairwise_cosine.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

In [ ]:
# ============================================================
# §3c (cont.) — cumulative explained variance curves
# ============================================================
# Figure 2: cumulative fraction of variance vs. number of components.
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, phrase in zip(axes, ["f_common_wndef_val", "f_common_wnex_val"]):
    for model, color in [("g_stock", "tab:blue"), ("g1", "tab:orange")]:
        cum = variance_detail[(model, phrase)]["_cum"]
        eff = variance_detail[(model, phrase)]["eff_dim"]
        ax.plot(np.arange(1, len(cum) + 1), cum, color=color,
                label=f"{model} (eff. dim={eff:.1f})")
    ax.set_xscale("log")
    ax.set_title(f"Cumulative variance — {phrase}")
    ax.set_xlabel("number of components (log scale)")
    ax.set_ylabel("fraction of total variance")
    ax.set_ylim(0, 1.02)
    ax.axhline(1.0, color="grey", linewidth=0.5)
    ax.legend(loc="lower right")
fig.suptitle("Effective dimensionality: how many components carry the variance?")
fig.tight_layout()
fig_path = FIG_DIR / "05_collapse_singular_values.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §4 — T=0 and T=1 similarity distributions

The ATE is a difference:
`ATE = mean( cos(g(f_clue(def)), g(f(ans))) − cos(g(f(def)), g(f(ans))) )`.

Looking only at the difference hides whether a change in ATE comes from
T=0 shifting, T=1 shifting, or both. In the NB 09 analysis we saw T=0
(the decontextualized similarity) rise while T=1 stayed roughly put — the
symptom of indiscriminate compression. Here we compute T=0 and T=1
separately for every evaluation pair and compare their distributions under
`g_stock` and `g1`.

**Evaluation pairs.** For each row of `clues_val.csv` we need three lookups:
the clue's `f_clue_val` row, the `f_common_wndef_val` row for
`definition_wn`, and the `f_common_wndef_val` row for `answer_wn`. A row
drops out if any of the three lookups fails (typically because the word
did not have a valid `f_common_wndef` phrase). We report the dropout
breakdown so the coverage is visible, not just implied.

In [ ]:
# ============================================================
# §4a — Assemble (clue, definition, answer) evaluation pairs
# ============================================================
t0 = time.time()

clue_rows_out = []
def_rows_out  = []
ans_rows_out  = []

# Separate counts for each dropout cause, to distinguish "definition missing"
# from "answer missing" from "clue missing from index".
n_total        = len(clues_val)
n_missing_clue = 0
n_missing_def  = 0
n_missing_ans  = 0

for cid, original_def, def_wn, ans_wn in zip(
    clues_val["clue_id"], clues_val["definition"],
    clues_val["definition_wn"], clues_val["answer_wn"]
):
    # f_clue is indexed by the ORIGINAL definition string (not definition_wn),
    # matching how f_clue.csv was built in NB 02.
    clue_row = clue_key_to_row.get((cid, original_def), -1)
    def_row  = wndef_word_to_row.get(def_wn, -1)
    ans_row  = wndef_word_to_row.get(ans_wn, -1)
    if clue_row < 0:
        n_missing_clue += 1
        continue
    if def_row < 0:
        n_missing_def += 1
        continue
    if ans_row < 0:
        n_missing_ans += 1
        continue
    clue_rows_out.append(clue_row)
    def_rows_out.append(def_row)
    ans_rows_out.append(ans_row)

clue_rows_arr = np.array(clue_rows_out, dtype=np.int64)
def_rows_arr  = np.array(def_rows_out,  dtype=np.int64)
ans_rows_arr  = np.array(ans_rows_out,  dtype=np.int64)
n_kept = len(clue_rows_arr)
assemble_seconds = time.time() - t0

print(f"clues_val rows:                      {n_total:,}")
print(f"  dropped (clue_id,def) not in index: {n_missing_clue:,}")
print(f"  dropped (definition_wn not in vocab_wndef_val): {n_missing_def:,}")
print(f"  dropped (answer_wn not in vocab_wndef_val):     {n_missing_ans:,}")
print(f"  kept (all three lookups resolved): {n_kept:,} "
      f"({n_kept/n_total:.1%})")
print(f"Assembly: {assemble_seconds:.1f}s")

In [ ]:
# ============================================================
# §4b — T=0 and T=1 similarities, per model
# ============================================================
def t0_t1(model):
    """Compute T=0 and T=1 per evaluation pair under one model."""
    clue_emb  = embeddings[(model, "f_clue_val")]
    wndef_emb = embeddings[(model, "f_common_wndef_val")]

    def_embeds = wndef_emb[def_rows_arr]
    ans_embeds = wndef_emb[ans_rows_arr]
    clue_embeds = clue_emb[clue_rows_arr]

    # T=0: decontextualized definition vs decontextualized answer.
    T0 = rowwise_cosine(def_embeds, ans_embeds)
    # T=1: clue-contextualized definition vs decontextualized answer.
    T1 = rowwise_cosine(clue_embeds, ans_embeds)
    return T0, T1

stock_T0, stock_T1 = t0_t1("g_stock")
g1_T0,    g1_T1    = t0_t1("g1")

def dist_stats(name, vec):
    return {
        "Distribution": name,
        "Mean":   float(vec.mean()),
        "Median": float(np.median(vec)),
        "Std":    float(vec.std()),
        "P5":     float(np.percentile(vec, 5)),
        "P95":    float(np.percentile(vec, 95)),
    }

t0_t1_table = pd.DataFrame([
    dist_stats("g_stock T=0", stock_T0),
    dist_stats("g_stock T=1", stock_T1),
    dist_stats("g1 T=0",      g1_T0),
    dist_stats("g1 T=1",      g1_T1),
])

with pd.option_context("display.float_format", "{:.4f}".format,
                       "display.width", 140):
    print(t0_t1_table.to_string(index=False))

# Also print the two per-model ATE means as a forward-reference — Stage 6 will
# do the full hypothesis testing. Keeping a pointer here prevents a reader from
# mentally "subtracting" the wrong numbers from the table above.
print(f"\nATE preview (mean of T=1 - T=0 per pair — formal test deferred to Stage 6):")
print(f"  g_stock ATE: {float((stock_T1 - stock_T0).mean()):+.4f}")
print(f"  g1 ATE:      {float((g1_T1    - g1_T0   ).mean()):+.4f}")

In [ ]:
# ============================================================
# §4c — Distribution visualization
# ============================================================
# Two panels side-by-side: g_stock on the left, g1 on the right. Within each
# panel we overlay T=0 and T=1, with vertical lines at each distribution's
# mean. The visual question is: did T=0 shift up while T=1 stayed put (the
# NB 09 pathology), or did they shift together (healthier)?
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

# Shared bin edges across all four distributions so the histograms are
# directly comparable both within and across panels.
all_vals = np.concatenate([stock_T0, stock_T1, g1_T0, g1_T1])
bins = np.linspace(all_vals.min(), all_vals.max(), 70)

panels = [
    (axes[0], "g_stock", stock_T0, stock_T1),
    (axes[1], "g1",      g1_T0,    g1_T1),
]
for ax, model, T0, T1 in panels:
    ax.hist(T0, bins=bins, alpha=0.55, color="tab:green",
            label=f"T=0 (def vs ans), mean={T0.mean():.3f}")
    ax.hist(T1, bins=bins, alpha=0.55, color="tab:purple",
            label=f"T=1 (clue-def vs ans), mean={T1.mean():.3f}")
    ax.axvline(T0.mean(), color="tab:green",  linewidth=1.2, linestyle=":")
    ax.axvline(T1.mean(), color="tab:purple", linewidth=1.2, linestyle=":")
    ax.set_title(f"{model} — T=0 vs T=1 ({n_kept:,} eval pairs)")
    ax.set_xlabel("cosine similarity")
    ax.set_ylabel("count")
    ax.legend()
fig.tight_layout()
fig_path = FIG_DIR / "05_t0_t1_distributions.png"
fig.savefig(fig_path, dpi=300)
plt.show()
print(f"Saved {fig_path}")

---

## §5 — Representational Similarity Analysis (RSA)

Collapse detection in §3 looks at marginal properties of the embedding
space (mean similarity, variance spread). RSA asks a relational question:
did the *shape* of the similarity structure survive fine-tuning? If pairs
of words that were similar under `g_stock` are still the most similar
pairs under `g1` — even if all similarities have shifted up or down — the
Spearman correlation of the two pairwise-similarity vectors will be near
1. If fine-tuning fundamentally reorganized which words are similar to
which, the correlation will be low.

Combining §3 and §5 gives a two-axis reading:

- **Low RSA rho + collapse signals** = the space compressed indiscriminately
- **Low RSA rho + healthy dimensionality** = the space reorganized structurally
- **High RSA rho** = fine-tuning was a global nudge, not a restructuring

We sample 1,000 words per phrase type (random_state=42) to keep the
1000×1000 cosine matrix small and the Spearman correlation fast. For the
wnex vocabulary (3,008 words) 1,000 is about a third of the words; for
wndef (26,152) it is less than 4%.

In [ ]:
# ============================================================
# §5 — Representational Similarity Analysis
# ============================================================
N_RSA = 1_000

def upper_triangle(mat):
    """Flatten the strict upper triangle of a square matrix."""
    n = mat.shape[0]
    iu = np.triu_indices(n, k=1)
    return mat[iu]

def pairwise_cosine_matrix(emb):
    """Return full NxN cosine similarity matrix for the given rows."""
    # Normalize once, then matmul — cheaper than rowwise-cosine over N^2 pairs.
    norms = np.linalg.norm(emb, axis=1, keepdims=True) + 1e-10
    normed = emb / norms
    return normed @ normed.T

rsa_rows = []
for phrase in ["f_common_wndef_val", "f_common_wnex_val"]:
    n_rows_phrase = embeddings[(MODEL_NAMES[0], phrase)].shape[0]
    # Sample the same row indices for both models (fixed seed per phrase).
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(n_rows_phrase, size=N_RSA, replace=False)

    # Build pairwise similarity matrix under each model, extract upper
    # triangle, then Spearman-correlate the two vectors.
    stock_mat = pairwise_cosine_matrix(embeddings[("g_stock", phrase)][sample_idx])
    g1_mat    = pairwise_cosine_matrix(embeddings[("g1",      phrase)][sample_idx])

    stock_ut = upper_triangle(stock_mat)
    g1_ut    = upper_triangle(g1_mat)
    rho, pval = spearmanr(stock_ut, g1_ut)

    rsa_rows.append({
        "Phrase type":      phrase,
        "N words sampled":  N_RSA,
        "N pair values":    len(stock_ut),
        "Spearman rho":     float(rho),
        "p-value":          float(pval),
    })

rsa_df = pd.DataFrame(rsa_rows)
with pd.option_context("display.float_format", "{:.6f}".format,
                       "display.width", 140):
    print(rsa_df.to_string(index=False))

---

## §6 — Write results file

All key numbers are serialized to `outputs/05_model_evaluation-results.md`
so the Architect can review Stage 5 without re-running the notebook. The
file is strictly a summary — no interpretation, no PASS/FAIL verdicts.
Interpretation happens during the Stage 5 review and in the Stage 6
hypothesis-testing notebook that follows.

In [ ]:
# ============================================================
# Build outputs/05_model_evaluation-results.md
# ============================================================
def _fmt_cell(v, float_fmt="{:.4f}"):
    if isinstance(v, (float, np.floating)):
        return float_fmt.format(v)
    return str(v)

def df_to_md(df, float_fmt="{:.4f}"):
    """Render a DataFrame as a GitHub-flavored markdown table."""
    cols = list(df.columns)
    header = "| " + " | ".join(cols) + " |"
    sep    = "|" + "|".join(["---"] * len(cols)) + "|"
    rows = [
        "| " + " | ".join(_fmt_cell(v, float_fmt) for v in row) + " |"
        for row in df.itertuples(index=False, name=None)
    ]
    return chr(10).join([header, sep, *rows])

lines = []
lines.append("# Results: 05 — Model Evaluation (Pre-Hypothesis Testing)\n")
lines.append(f"**Date:** {date.today().isoformat()}  ")
lines.append(f"**Environment:** {env_label}\n")

lines.append("## Versions\n")
lines.append(f"- pandas:     {pd.__version__}")
lines.append(f"- numpy:      {np.__version__}")
lines.append(f"- scipy:      {scipy.__version__}")
lines.append(f"- matplotlib: {matplotlib.__version__}")
lines.append(f"- seaborn:    {seaborn.__version__}\n")

lines.append("## Scope\n")
lines.append("Canonical mean-pooling models only: `g_stock` and `g1`. "
             "The `_tokenspan` variants are out of scope per Decision 20.\n")

# §2
lines.append("## §2 — Validation triplet accuracy\n")
lines.append(f"Validation triplet file: `data/triplets/g1_val.csv` "
             f"({n_total:,} rows).")
lines.append(f"- anchors resolved:   {n_total - n_miss_anchor:,} / {n_total:,}")
lines.append(f"- positives resolved: {n_total - n_miss_positive:,} / {n_total:,}")
lines.append(f"- negatives resolved: {n_total - n_miss_negative:,} / {n_total:,}")
lines.append(f"- all three resolved: {n_valid:,} / {n_total:,} "
             f"({n_valid / n_total:.1%}) — used for the accuracy table below")
if n_miss_negative > 0:
    lines.append(f"\nNote: {n_miss_negative:,} distractor_wn values are absent "
                 f"from vocabulary_wndef_val.csv (validation-split wndef vocabulary). "
                 f"Distractors are drawn from the full WordNet vocabulary; only "
                 f"those also present in the validation split can be resolved to "
                 f"a g1 / g_stock validation embedding. See DECISIONS.md "
                 f"Decision 21 and "
                 f"`planning/questions/05_model_evaluation-questions.md`.")
    lines.append("")
    lines.append("**Bias caveat** (per DECISIONS.md Decision 21): the surviving "
                 "triplets' negatives are words that also appear as definitions "
                 "or answers in validation clues — i.e., common crossword words. "
                 "Whether these are systematically easier or harder negatives "
                 "than the dropped distractors is unknown. However, the "
                 "comparison between `g_stock` and `g1` is computed on the "
                 "identical set of triplets, so any difficulty bias affects both "
                 "models equally and does not compromise the `g_stock`-vs-`g1` "
                 "comparison.\n")
else:
    lines.append("")
lines.append(df_to_md(triplet_table))
lines.append("")
lines.append(f"Figure: `outputs/figures/05_val_triplet_accuracy.png`\n")

# §3a
lines.append("## §3a — Mean pairwise cosine among random word pairs\n")
lines.append(f"Sampled {N_PAIRS:,} random distinct-row pairs per "
             f"(model, phrase) with random_state={RANDOM_STATE}. Same pairs "
             f"used for both models within a phrase type.\n")
lines.append(df_to_md(pairwise_df))
lines.append("")

# §3b
lines.append("## §3b — Embedding variance and effective dimensionality\n")
lines.append(df_to_md(variance_df, float_fmt="{:.2f}"))
lines.append("")
lines.append("Figures: `outputs/figures/05_collapse_pairwise_cosine.png`, "
             "`outputs/figures/05_collapse_singular_values.png`\n")

# §4
lines.append("## §4 — T=0 and T=1 similarity distributions\n")
lines.append(f"Evaluation pairs assembled from clues_val.csv ({n_total:,} rows):")
lines.append(f"- dropped: (clue_id, definition) not in f_clue index: {n_missing_clue:,}")
lines.append(f"- dropped: definition_wn not in vocabulary_wndef_val:   {n_missing_def:,}")
lines.append(f"- dropped: answer_wn not in vocabulary_wndef_val:       {n_missing_ans:,}")
lines.append(f"- kept:   {n_kept:,} ({n_kept/n_total:.1%})\n")
lines.append(df_to_md(t0_t1_table))
lines.append("")
lines.append(f"ATE preview (deferred to Stage 6):")
lines.append(f"- g_stock ATE (mean of T=1 - T=0): {float((stock_T1 - stock_T0).mean()):+.4f}")
lines.append(f"- g1 ATE      (mean of T=1 - T=0): {float((g1_T1    - g1_T0   ).mean()):+.4f}\n")
lines.append(f"Figure: `outputs/figures/05_t0_t1_distributions.png`\n")

# §5
lines.append("## §5 — RSA (Spearman correlation of pairwise cosines)\n")
lines.append(df_to_md(rsa_df, float_fmt="{:.6f}"))
lines.append("")

results_path = OUTPUT_DIR / "05_model_evaluation-results.md"
results_path.write_text(chr(10).join(lines))
print(f"Wrote {results_path}")

---

## Summary

This notebook produced four diagnostic readings of `g1` before Stage 6
hypothesis testing:

1. **Validation triplet accuracy (§2)** — Does `g1` satisfy the triplet
   constraint on held-out rows that were never part of training? Reported
   alongside `g_stock`'s baseline triplet accuracy; the gap between them
   is the generalization signal.
2. **Collapse detection (§3)** — Mean pairwise cosine among random word
   pairs, plus the participation ratio and cumulative-variance curves.
   Diagnoses indiscriminate compression (the NB 09 failure mode) vs.
   healthy fine-tuning.
3. **T=0 and T=1 distributions (§4)** — Decomposes the two components of
   the ATE so we can see whether a change in ATE is driven by the
   decontextualized baseline (T=0) shifting, the clue-contextualized
   treatment (T=1) shifting, or both. An ATE preview is printed, but the
   formal hypothesis test is Stage 6's responsibility.
4. **RSA (§5)** — Single-number measure of how much the similarity
   structure reorganized. Combined with §3 it distinguishes "compressed
   but structure preserved" from "structurally reorganized" from "global
   shift only."

**Outputs:**
- `outputs/05_model_evaluation-results.md` — all numerical results
- `outputs/figures/05_val_triplet_accuracy.png`
- `outputs/figures/05_collapse_pairwise_cosine.png`
- `outputs/figures/05_collapse_singular_values.png`
- `outputs/figures/05_t0_t1_distributions.png`

No new data artifacts are written — the notebook is read-only with respect
to the data directory.

**Runtime:** A few seconds on CPU, dominated by the two SVDs over the
26,152-row `f_common_wndef_val` matrices in §3b.